# Preporuke utakmica

In [1]:
import os
import time
import numpy as np
from dotenv import load_dotenv
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from clickhouse_driver import Client

load_dotenv()

client = Client(
    host=os.environ.get("CH_HOST", "clickhouse.sofascore.ai"),
    port=9000,
    user=os.environ.get("CH_USER"),
    password=os.environ.get("CH_PASS"),
)

In [2]:
ukupno, bez_acc, bez_timova, kasni, krivi_mcc = client.execute("""
SELECT count(),
  countIf(user_account_id IS NULL),
  countIf(empty(teams)),
  countIf(created_at >= '2024-10-01' OR updated_at >= '2024-10-01'),
  countIf(mcc IS NULL OR mcc NOT IN (216, 218, 219, 220, 221, 222, 226, 232, 262, 276, 284, 293, 294, 297))
FROM bq.mobile_user
""")[0]
print("redaka ukupno", ukupno)
print("bez accounta", bez_acc)
print("bez timova", bez_timova)
print("izvan granica", kasni)
print("mcc izvan liste", krivi_mcc)

redaka, racuna = client.execute("""
SELECT count(), uniqExact(user_account_id)
FROM bq.mobile_user
WHERE user_account_id IS NOT NULL
  AND notEmpty(teams)
  AND created_at < '2024-10-01' AND updated_at < '2024-10-01'
  AND mcc IN (216, 218, 219, 220, 221, 222, 226, 232, 262, 276, 284, 293, 294, 297)
""")[0]
print("valjani", redaka)
print("jedinstveni", racuna)

redaka ukupno 18606517
bez accounta 15966627
bez timova 9350814
izvan granica 10352260
mcc izvan liste 16444516
valjani 51756
jedinstveni 46949


In [3]:
def dohvati_timove():
    return {r[0] for r in client.execute("""
    SELECT DISTINCT arrayJoin([hometeam_id, awayteam_id])
    FROM sports.event
    WHERE sport_id = (SELECT id FROM sports.sport WHERE lower(name) = 'football')
        AND startdate >= '2023-01-01' AND startdate < '2023-07-01'
    """)}


def dohvati_usere():
    return client.execute("""
    SELECT user_account_id, teams, mcc
    FROM bq.mobile_user
    WHERE user_account_id IS NOT NULL
        AND notEmpty(teams)
        AND created_at < '2024-10-01' AND updated_at < '2024-10-01'
        AND mcc IN (216, 218, 219, 220, 221, 222, 226, 232, 262, 276, 284, 293, 294, 297)
    ORDER BY mcc
    """)


def dohvati_evente():
    eventi = {}
    q = client.execute("""
    SELECT toMonday(startdate), id, hometeam_id, awayteam_id
    FROM sports.event
    WHERE sport_id = (SELECT id FROM sports.sport WHERE lower(name) = 'football')
        AND startdate >= '2023-06-01' AND startdate < '2023-07-01'
    ORDER BY toMonday(startdate)
    """)
    for ws, eid, h, a in q:
        eventi.setdefault(ws, []).append((eid, h, a))
    return eventi

def dohvati_mcc(timovi):
    S = {}
    for acc, prac, mcc in dohvati_usere():
        if acc in S:
            S[acc][0].update(prac)
        else:
            S[acc] = (set(prac), mcc)
    po_mcc = {}
    for acc, (prac, mcc) in S.items():
        if prac & timovi:
            po_mcc.setdefault(mcc, []).append((acc, prac & timovi))
    return po_mcc

In [4]:
timovi = dohvati_timove()
eventi = dohvati_evente()
po_mcc = dohvati_mcc(timovi)

print("timova", len(timovi))
print("usera", sum(len(v) for v in po_mcc.values()))
for mcc in sorted(po_mcc):
    print("mcc", mcc, "usera", len(po_mcc[mcc]))

print("tjedana", len(eventi))
for ws in sorted(eventi):
    print("tjedan", ws, "eventa", len(eventi[ws]))

vel = [len(p) for v in po_mcc.values() for _, p in v]
print("pracenih timova po useru", np.percentile(vel, [20, 50, 90, 99]))

timova 21274
usera 43696
mcc 216 usera 1874
mcc 218 usera 2746
mcc 219 usera 7134
mcc 220 usera 5514
mcc 221 usera 169
mcc 222 usera 13746
mcc 226 usera 2947
mcc 232 usera 976
mcc 262 usera 4885
mcc 276 usera 702
mcc 284 usera 1528
mcc 293 usera 754
mcc 294 usera 241
mcc 297 usera 480
tjedana 5
tjedan 2023-05-29 eventa 3148
tjedan 2023-06-05 eventa 2736
tjedan 2023-06-12 eventa 1863
tjedan 2023-06-19 eventa 1841
tjedan 2023-06-26 eventa 841
pracenih timova po useru [ 1.  3. 18. 91.]


## Task 2

In [5]:
def izgradi_matricu(po_mcc, timovi):
    tim_idx = {t: j for j, t in enumerate(sorted(timovi))}
    accs, mccs = [], []
    redovi, stupci = [], []
    r = 0
    for mcc in sorted(po_mcc):
        for acc, prac in po_mcc[mcc]:
            for t in prac:
                j = tim_idx.get(t)
                if j is not None:
                    redovi.append(r)
                    stupci.append(j)
            accs.append(acc)
            mccs.append(mcc)
            r += 1
    U = csr_matrix((np.ones(len(redovi)), (redovi, stupci)), shape=(r, len(tim_idx)))
    return U, np.array(accs, dtype=object), np.array(mccs), tim_idx

def event_matrica(eventi_tj, tim_idx, n_tim):
    redovi, stupci = [], []
    for c, (_, h, a) in enumerate(eventi_tj):
        for t in (h, a):
            j = tim_idx.get(t)
            if j is not None:
                redovi.append(j)
                stupci.append(c)
    return csr_matrix((np.ones(len(redovi)), (redovi, stupci)), shape=(n_tim, len(eventi_tj)))

def topk_cosine(X, Y, k):
    Xn = normalize(X)
    Yn = normalize(Y)
    out = np.empty((X.shape[0], k), dtype=np.int64)
    for s in range(0, X.shape[0], 1024):
        sim = (Xn[s:s + 1024] @ Yn.T).toarray()
        part = np.argpartition(-sim, kth=k - 1, axis=1)[:, :k]
        rows = np.arange(part.shape[0])[:, None]
        ordd = np.argsort(-sim[rows, part], axis=1)
        out[s:s + 1024] = part[rows, ordd]
    return out


def topk_jaccard(X, Y, k):
    vely = np.asarray(Y.sum(axis=1)).ravel()
    out = np.empty((X.shape[0], k), dtype=np.int64)
    for s in range(0, X.shape[0], 1024):
        inter = (X[s:s + 1024] @ Y.T).toarray()
        velx = np.asarray(X[s:s + 1024].sum(axis=1)).ravel()
        union = velx[:, None] + vely[None, :] - inter
        sim = inter / np.maximum(union, 1)
        part = np.argpartition(-sim, kth=k - 1, axis=1)[:, :k]
        rows = np.arange(part.shape[0])[:, None]
        ordd = np.argsort(-sim[rows, part], axis=1)
        out[s:s + 1024] = part[rows, ordd]
    return out


def preporuke(U, mccs, tim_idx, eventi, topk=topk_cosine):
    k, n_pop = 10, 10
    n_tim = U.shape[1]
    n_users = U.shape[0]
    U = U.tocsr()
    prep = {}
    for ws in sorted(eventi):
        eventi_tj = eventi[ws]
        eids = [e[0] for e in eventi_tj]
        n_ev = len(eids)
        T = event_matrica(eventi_tj, tim_idx, n_tim)
        M = (U @ T).tocsr()
        Rb = (M > 0).tocsr()
        ima = np.diff(M.indptr) > 0
        pop = np.asarray(Rb.sum(axis=0)).ravel()
        top_idx = np.argsort(pop)[::-1][:n_pop]
        nema = np.where(~ima)[0]
        redovi, stupci, vrijednosti = [], [], []
        for mcc in np.unique(mccs[nema]):
            norec = nema[mccs[nema] == mcc]
            kand = np.where(ima & (mccs == mcc))[0]
            if len(kand) == 0:
                redovi.append(np.repeat(norec, len(top_idx)))
                stupci.append(np.tile(top_idx, len(norec)))
                vrijednosti.append(np.tile(pop[top_idx], len(norec)))
                continue
            kk = min(k, len(kand))
            sig = {}
            inv = np.empty(len(norec), dtype=np.int64)
            reps = []
            for i, u in enumerate(norec):
                kljuc = U.indices[U.indptr[u]:U.indptr[u + 1]].tobytes()
                j = sig.get(kljuc)
                if j is None:
                    j = len(reps)
                    sig[kljuc] = j
                    reps.append(u)
                inv[i] = j
            reps = np.array(reps)
            susjedi = topk(U[reps], U[kand], kk)
            S = csr_matrix((np.ones(susjedi.size),
                            (np.repeat(np.arange(len(reps)), kk), susjedi.ravel())),
                            shape=(len(reps), len(kand)))
            Fu = (S @ Rb[kand]).tocsr()
            for i, u in enumerate(norec):
                s0, e0 = Fu.indptr[inv[i]], Fu.indptr[inv[i] + 1]
                cols = Fu.indices[s0:e0]
                redovi.append(np.full(len(cols), u))
                stupci.append(cols)
                vrijednosti.append(Fu.data[s0:e0])
        if redovi:
            redovi = np.concatenate(redovi)
            stupci = np.concatenate(stupci)
            vrijednosti = np.concatenate(vrijednosti)
            M = (M + csr_matrix((vrijednosti, (redovi, stupci)), shape=(n_users, n_ev))).tocsr()
        prep[ws] = (M, eids)
    return prep

def preporuke_usera(prep, u):
    W, eids = prep
    s, e = W.indptr[u], W.indptr[u + 1]
    cols = W.indices[s:e]
    poredak = np.argsort(-W.data[s:e], kind="stable")
    return [eids[c] for c in cols[poredak]]

In [6]:
U, accs, mccs, tim_idx = izgradi_matricu(po_mcc, timovi)

t = time.time()
prep = preporuke(U, mccs, tim_idx, eventi)
print("vrijeme", round(time.time() - t, 2))
print("usera", U.shape[0], "tjedana", len(prep), "praznih", sum(int((np.diff(W.indptr) == 0).sum()) for W, _ in prep.values()))

vrijeme 2.33
usera 43696 tjedana 5 praznih 0


In [7]:
prvi = sorted(eventi)[0]

t = time.time()
a = preporuke(U, mccs, tim_idx, {prvi: eventi[prvi]}, topk=topk_cosine)
ta = time.time() - t
t = time.time()
b = preporuke(U, mccs, tim_idx, {prvi: eventi[prvi]}, topk=topk_jaccard)
tb = time.time() - t

Wa, _ = a[prvi]
Wb, _ = b[prvi]
D = (Wa > 0).astype(np.int8) - (Wb > 0).astype(np.int8)
print("cosine", round(ta, 2), "jaccard", round(tb, 2))
print("udio razlicitih", round(len(np.unique(D.nonzero()[0])) / U.shape[0], 4))

cosine 0.28 jaccard 0.48
udio razlicitih 0.0916


## Provjera

In [8]:
for ws in sorted(eventi):
    T = event_matrica(eventi[ws], tim_idx, U.shape[1])
    ima = np.diff(((U @ T) > 0).tocsr().indptr) > 0
    broj = np.diff((prep[ws][0] > 0).tocsr().indptr)
    assert broj.min() >= 1, ws
    print(ws, "usera", U.shape[0], "direktno", int(ima.sum()),
        "fallback", int((~ima).sum()), "min", int(broj.min()),
        "prosjek", round(float(broj.mean()), 1))

idx_tim = np.array(sorted(tim_idx))
ws = sorted(eventi)[0]
ev = {e: (h, a) for e, h, a in eventi[ws]}
D = ((U @ event_matrica(eventi[ws], tim_idx, U.shape[1])) > 0).tocsr()
W = (prep[ws][0] > 0).tocsr()
ima = np.diff(D.indptr) > 0

ud = int(np.where(ima)[0][0])
uf = int(np.where(~ima & np.isin(mccs, np.unique(mccs[ima])))[0][0])
kand = np.where(ima & (mccs == mccs[uf]))[0]
glob = kand[topk_cosine(U[[uf]], U[kand], min(10, len(kand)))[0]]

prati = lambda u: idx_tim[U.indices[U.indptr[u]:U.indptr[u + 1]]]
rec = lambda u: preporuke_usera(prep[ws], u)[:5]
ids = set(prati(ud)) | set(prati(uf)) | {t for u in (ud, uf) for e in rec(u) for t in ev[e]}
s = ",".join(map(str, sorted({int(x) for x in ids})))
nm = dict(client.execute(f"SELECT id, name FROM sports.team WHERE id IN ({s})"))
mec = lambda e: f"{nm.get(ev[e][0], ev[e][0])} vs {nm.get(ev[e][1], ev[e][1])}"
unija = set().union(*(set(D.indices[D.indptr[v]:D.indptr[v + 1]]) for v in glob))

print("tjedan", ws)
print("direktan", accs[ud], "prati", [nm.get(t, int(t)) for t in prati(ud)])
for e in rec(ud):
    print("  ", mec(e))
print("fallback", accs[uf], "10 susjeda", [accs[g] for g in glob])
for e in rec(uf):
    print("  ", mec(e))
print("preporuke kroz susjede", unija == set(W.indices[W.indptr[uf]:W.indptr[uf + 1]]))


2023-05-29 usera 43696 direktno 32218 fallback 11478 min 1 prosjek 7.5
2023-06-05 usera 43696 direktno 19776 fallback 23920 min 1 prosjek 7.9
2023-06-12 usera 43696 direktno 27518 fallback 16178 min 1 prosjek 5.7
2023-06-19 usera 43696 direktno 25440 fallback 18256 min 1 prosjek 7.7
2023-06-26 usera 43696 direktno 17614 fallback 26082 min 1 prosjek 6.9
tjedan 2023-05-29
direktan 5a15cb1c1e6bce5f29872da3 prati ['FC Bayern München', 'Juventus', 'Hungary']
   Udinese vs Juventus
fallback 6673b423e4069a086415de34 10 susjeda ['5fa310e9ccf0be014dfeb45b', '61cfb33b6d0e99d073ef5fcf', '5aa1aee86b323cb83716eaf1', '55d9f652731593d916e716b6', '5f2f55ebb20d51d53a3435ce', '66730a76e63867b4b5bdc3fc', '661eee89fe0bec8e57b969e7', '5c5b4d014a0a310cb054dc7d', '6640fe2be55ef54cde145213', '57962307770c4b2c3d3a5d85']
   Celta Vigo vs Barcelona
   Real Madrid vs Athletic Club
   Milan vs Hellas Verona
   Udinese vs Juventus
   Paris Saint-Germain vs Clermont Foot
preporuke kroz susjede True


## Zakljucak

Svaki user dobiva rangiranu listu preporuka u svakom tjednu, direktno preko timova koje prati ili preko najslicnijih usera iz iste drzave kad mu timovi taj tjedan ne igraju, ako u drzavi nema takvih usera, koriste se globalno najpraceniji mecevi (u nasoj bazi svaki fallback user ih je imao). Rrangiranje dolazi besplatno iz matrice rezultata jer je rezultat broj pracenih timova u mecu. Cosine i Jaccard daju gotovo iste liste, a cosine je zadrzan kao brza varijanta. Najvise vremena traje dohvat podataka, dok generiranje preporuka za cijeli mjesec ostaje na par sekundi zahvaljujuci rijetkim matricama i racunanju slicnosti samo po jedinstvenim skupovima pracenih timova.